**Imports**

In [ ]:
%matplotlib inline


In [ ]:
import os
from pathlib import Path

In [ ]:
import SimpleITK as sitk
import nibabel as nib
import numpy as np
from scipy import ndimage

In [ ]:
from utils.utils import initialization_dict, get_lesion_track, improved_cca, preprocess_mask
from utils.graphics import explore_3D_array, explore_patient_timepoints, visualize_matching

**Dictionary - File system**

In [ ]:
base_path = Path("dataset/")

dataset_dict = initialization_dict(base_path)

In [ ]:
patient_id = 'P5'

sample_file_path = str(dataset_dict[patient_id]['T1']['T2'])
sitk_image = sitk.ReadImage(sample_file_path)
image_array = sitk.GetArrayFromImage(sitk_image)
print(f"Размерность массива: {image_array.shape}")

In [ ]:
explore_3D_array(image_array)

**MASKS**

In [ ]:
patient_data = dataset_dict[patient_id]
timepoints = sorted(patient_data.keys())

masks = {}
for tp in timepoints:
    key = next((k for k in patient_data[tp] if k.upper() == "MASK"), None)
    if key:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(str(patient_data[tp][key])))
        masks[tp] = (arr > 0).astype(np.uint8)

print(masks.keys())


In [ ]:
masks = {tp: preprocess_mask(masks[tp]) for tp in timepoints}   # mask preprocessing

In [ ]:
explore_patient_timepoints(dataset_dict, patient_id=patient_id, modality='MASK', cmap='gray')

In [ ]:
labeled = {}
n_components = {}

for tp, mask in masks.items():
    lbl, n = ndimage.label(masks[tp])
    #lbl, n = improved_cca(masks[tp], max_filter_size=20)
    labeled[tp] = lbl
    n_components[tp] = n

    total_voxels = (lbl > 0).sum()

    print(f'{tp}: finded components = {n}, voxels: {total_voxels}')


In [ ]:
masks.keys()

In [ ]:
tracks = get_lesion_track(timepoints, labeled, n_components)

for track in tracks.values():
    print(" -> ".join(map(str, track)))

In [ ]:
matched_t2_ids = set(
    track[-1] for track in tracks.values() if track[-1] is not None
)
all_t2_ids = set(range(1, n_components[timepoints[-1]] + 1))
unmatched_t2 = all_t2_ids - matched_t2_ids

print("T2 lesions without pair from T1:", unmatched_t2)
for comp_id in unmatched_t2:
    size = (labeled[timepoints[-1]] == comp_id).sum()
    center = ndimage.center_of_mass(labeled[timepoints[-1]] == comp_id)
    print(f"T2 id={comp_id}: voxels={size}")

print("\nT1 disappeared lesions:")
for track_id, track in tracks.items():
    if track[-1] is None:
        comp_id = track[0]
        size = (labeled[timepoints[0]] == comp_id).sum()
        center = ndimage.center_of_mass(labeled[timepoints[0]] == comp_id)
        print(f"T1 id={comp_id}: voxels={size}")

In [ ]:
count_not_None = 0
for i in tracks.values():
    if i[1] != None:
        count_not_None += 1
print(count_not_None / len(tracks) * 100)

In [ ]:
visualize_matching(labeled, tracks.values())